## 加载数据集
1. uv add datasets
2. 替换镜像源头
3. 导入斯坦福imdb数据集


In [1]:
from datasets import load_dataset
import os

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

train_dataset = load_dataset("stanfordnlp/imdb", split="train")
#test_dataset = load_dataset("stanfordnlp/imdb", split="test")

C:\Users\10115\PycharmProjects\cs224n\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 创建词表

In [2]:
import string

# 构建词表
# TODO词粒度word
'''
1. oov问题：没见过的词用索引0表示
2. 无法处理单词的形态和词缀关系：比如live,lives,living都会被认为是一种词
3. 长尾效应
'''
all_words = {
    word
    for i in range(train_dataset.num_rows)
    for word in train_dataset['text'][i]
    .translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    .lower()
    .split()
}
# 构建词到索引的映射
word2idx = {word: i + 1 for i, word in enumerate(all_words)}
word2idx['<unk>'] = 0


In [3]:
from torch.utils.data import Dataset, DataLoader


class IMDB_Dataset(Dataset):
    def __init__(self, split: str) -> None:
        self.dataset = load_dataset("stanfordnlp/imdb", split=split)

    def __len__(self) -> int:
        return self.dataset.num_rows

    #todo 可以重写，使其也支持key访问
    def __getitem__(self, idx: int) -> tuple[list[str], int]:
        text_idx = [
            word
            for word in self.dataset[idx]['text']
            .translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
            .lower().split()
        ]

        text_idx_label = self.dataset[idx]['label']
        return text_idx, text_idx_label



In [22]:
import torch
from torch import Tensor, nn
from torch.nn.utils.rnn import pad_sequence

# 建议在外部定义好 device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def collate_fn(batch: list[tuple[list[str], int]]) -> tuple[Tensor, Tensor]:
    """
    修正点 1: 返回值类型改为 tuple[Tensor, Tensor]
    因为 pad_sequence 之后，list 就变成了单一的 Tensor
    """
    text_list, label_list = [], []

    for _text, _label in batch:
        # 将词转为索引
        _text_idxs = [word2idx.get(word, 0) for word in _text]

        # 修正点 2: 暂时不要传 device=device
        # 先在 CPU 上创建 Tensor，聚合速度更快，且避免了频繁的 CPU-GPU 数据拷贝
        text_list.append(torch.tensor(_text_idxs, dtype=torch.int64))
        label_list.append(_label)

    # 修正点 3: 聚合数据
    # pad_sequence 接收 list[Tensor]，返回 Tensor
    padded_text = pad_sequence(text_list, batch_first=True, padding_value=0)

    # 转换标签
    label_tensor = torch.tensor(label_list, dtype=torch.int64)

    # 修正点 4: 统一移动到 GPU
    # 这样只进行一次 GPU 通信，效率最高
    return padded_text.to(device), label_tensor.to(device)

In [23]:
dataset = IMDB_Dataset("train")
test_dataset = IMDB_Dataset("test")
print(collate_fn([(dataset[0])]))

(tensor([[75183, 55544, 75183,  8942,  2154,  3981, 41424, 65078, 22513, 69597,
         43326, 23682, 73833, 15724, 37003, 49063,  1250, 18465,   626, 18465,
         10343,  9505, 26800, 23868, 30278, 75183, 41221, 46263, 49063, 53871,
          9505, 18465, 10343, 55299, 50417, 52449, 47564, 71046, 53820, 18465,
         26972, 57374, 13962, 37946, 29748, 72054, 10379, 37318, 67559, 47276,
         23682, 38163, 58142, 12968, 75183, 72865,  4530, 13962, 50373, 29748,
         19261, 29478, 52394, 52394, 15724, 69963, 44417, 68231, 60585, 67559,
         29100, 70302, 19590, 66837,  3339,  4839, 69096,  7893, 13962,   423,
          2666, 14678, 16941, 34431, 65050, 23868, 49872, 14678,  7893, 13962,
         65885, 64475, 72638, 13962, 46815,  4353, 61190, 23682,  8911,  7002,
         24394, 15724,  3038, 13204,  5047, 34431, 69983, 33009, 40623, 15948,
          6154, 15724, 62664, 20512, 11929, 59249, 40623, 23868, 15724,    70,
         64298, 23868, 15337, 25779, 26929, 11929, 

In [8]:
x = torch.tensor([1, 2, 3])
y = torch.tensor([1, 2])
text_list = [x, y]
torch.nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=0)

tensor([[1, 2, 3],
        [1, 2, 0]])

In [24]:
train_dataloader = DataLoader(dataset=dataset,
                              batch_size=128,
                              collate_fn=collate_fn)

test_dataloader = DataLoader(dataset=test_dataset,
                             batch_size=128,
                             collate_fn=collate_fn)

for i ,(text, label) in enumerate(train_dataloader):
    print(f"当前批次{i}，文本形状{text.shape},标签形状{label.shape}")
    if i  == 8:
        break


<class 'torch.Tensor'> torch.Size([128])
当前批次0，文本形状torch.Size([128, 930]),标签形状torch.Size([128])
当前批次1，文本形状torch.Size([128, 1024]),标签形状torch.Size([128])
当前批次2，文本形状torch.Size([128, 876]),标签形状torch.Size([128])
当前批次3，文本形状torch.Size([128, 1057]),标签形状torch.Size([128])
当前批次4，文本形状torch.Size([128, 1023]),标签形状torch.Size([128])
当前批次5，文本形状torch.Size([128, 1030]),标签形状torch.Size([128])
当前批次6，文本形状torch.Size([128, 1036]),标签形状torch.Size([128])
当前批次7，文本形状torch.Size([128, 1050]),标签形状torch.Size([128])
当前批次8，文本形状torch.Size([128, 1341]),标签形状torch.Size([128])


In [53]:
import numpy as np

class ManualBatchNorm:
    def __init__(self, num_features, momentum=0.9, eps=1e-5):
        # 可学习参数 (对应公式中的 gamma 和 beta)
        self.gamma = np.ones((1, num_features))
        self.beta = np.zeros((1, num_features))

        # 推理时使用的全局统计量 (Running statistics)
        self.running_mean = np.zeros((1, num_features))
        self.running_var = np.ones((1, num_features))

        self.momentum = momentum # 用于更新全局统计量的动量
        self.eps = eps           # 防止分母为 0
        self.training = True     # 状态开关

    def forward(self, x):
        """
        x 的维度通常为 (Batch_size, Features)
        如果是卷积层输出，通常需要先将 (N, C, H, W) 重塑为 (N*H*W, C)
        """
        if self.training:
            # 1. 计算当前 Batch 的均值和方差
            # 沿着 Batch 维度（axis=0）求均值
            batch_mean = np.mean(x, axis=0, keepdims=True)
            batch_var = np.var(x, axis=0, keepdims=True)

            # 2. 标准化 (Standardization)
            x_hat = (x - batch_mean) / np.sqrt(batch_var + self.eps)

            # 3. 更新全局统计量 (为推理做准备)
            # 使用指数移动平均 (EMA)
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * batch_mean
            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * batch_var
        else:
            # 推理模式：直接使用训练期间保存的全局统计量
            x_hat = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)

        # 4. 缩放与偏移 (Scale and Shift)
        out = self.gamma * x_hat + self.beta
        return out

# --- 测试代码 ---
# 假设有 3 个样本，每个样本 2 个特征
data = np.array([[1.0, 2.0],
                 [2.0, 4.0],
                 [3.0, 6.0]])

bn = ManualBatchNorm(num_features=2)

# 模拟训练阶段
bn.training = True
print("训练输出:\n", bn.forward(data))

# 模拟推理阶段 (输入同一个样本，看输出是否稳定)
bn.training = False
print("推理输出:\n", bn.forward(np.array([[1.0, 2.0]])))


(1, 2)
训练输出:
 [[-1.22473569 -1.22474258]
 [ 0.          0.        ]
 [ 1.22473569  1.22474258]]
推理输出:
 [[0.813672   1.48130581]]


In [63]:
x = torch.randn(4, 4)
x.size()
y = x.view(16)
y.size()
z = x.view(-1, 8)  # the size -1 is inferred from other dimensions
z.size()

a = torch.randn(1, 2, 3, 4)
a.size()
b = a.transpose(1, 2)  # Swaps 2nd and 3rd dimension
b.size()
c = a.view(1, 3, 2, 4)  # Does not change tensor layout in memory
c.size()
torch.equal(b, c)

False

In [64]:
x = torch.randn(3, 4)
print(x)
indices = torch.tensor([0, 2])
torch.index_select(x, 0, indices)
torch.index_select(x, 1, indices)

tensor([[ 1.5609,  0.2503,  0.8519, -0.7297],
        [-0.0282, -2.2045,  0.7066,  0.2602],
        [ 0.7858, -0.4662,  1.7945, -0.4245]])


tensor([[ 1.5609,  0.8519],
        [-0.0282,  0.7066],
        [ 0.7858,  1.7945]])

In [69]:
from torch import nn
m = nn.Dropout(p=0.2)
input = torch.randn(4, 4)
print(input)
output = m(input)
print(output)

tensor([[-0.5101,  1.6119, -0.1058,  0.8774],
        [-0.3424, -0.6677,  0.6751,  0.5815],
        [-0.1079, -0.5926,  1.1555, -0.3591],
        [-0.3751, -0.4079,  0.5135,  1.6882]])
tensor([[-0.6376,  2.0149, -0.1323,  1.0967],
        [-0.4280, -0.8346,  0.8438,  0.7268],
        [-0.1348, -0.0000,  0.0000, -0.4488],
        [-0.4689, -0.0000,  0.6419,  2.1102]])


In [81]:
x = torch.empty(4, 4)
y = np.zeros((4,3))
y1 = torch.from_numpy(y)
y1[0][1] = 2
print(y)
print(type(y1))
print(y1.detach().cpu().numpy())

[[0. 2. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
<class 'torch.Tensor'>
[[0. 2. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


In [82]:
from tqdm import tqdm
with tqdm(total=100) as pbar:
    for i in range(100):
        pbar.update(1)

100%|██████████| 100/100 [00:00<00:00, 99935.76it/s]
